# 01 — Problem and Data Understanding

## What this notebook covers
- The frame of business problem
- A look about what we have

### Business problem
A company wants to maximize the return on its marketing budget by identifying customers who are most likely to respond to a campaign.
The goal of this end-to-end data science pipeline is **to build a predicting system that will help the company target the clients who're most likely to respond to the campaign** and so, maximizing the marketing budget

### ML framing

| Element | Choice | Why |
|---|---|---|
| Unit of observation | One customer (`ID`) | One prediction per customer we may contact |
| Target | `Response` (1 = accepted the last campaign) | The event the business pays for |
| Task | Binary classification, output = **probability** | We need a ranking + an adjustable decision threshold |
| Primary metric | **PR-AUC** | Rare positive class; PR-AUC focuses on the positives |
| Secondary metrics | ROC-AUC, precision, recall, F1, lift @ top-k% | Ranking quality and targeting efficiency |
| Business metric | Profit = conversions × revenue − contacts × cost | The final threshold is chosen on profit |
| Metric to avoid | Accuracy | Predicting "nobody responds" already scores ~85% |

### Two datasets, two questions (never merged)
- **Marketing campaign** (customer level) → *who* should we target? (supervised classification)
- **A/B test** (user level) → does showing the ad *cause* more conversions? (randomised experiment)

A model captures correlation; only a randomised experiment measures a causal effect.

### Loading packages

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt 

PROJECT_ROOT = Path.cwd().parent        # .cwd() gives the directory notebooks/ and so .parent gives the root
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_marketing_campaign, load_ab_test
from src.config import TARGET, ID_COL, CATEGORICAL_COLS, PAST_CAMPAIGN_COLS, AB_GROUP_COL, AB_TARGET

In [2]:
marketing_df = load_marketing_campaign()

print("---Dataframe shape---")
marketing_df.shape

---Dataframe shape---


(2240, 29)

We're dealing here with 2240 clients 

In [3]:
print("---First 10 rows---")
marketing_df.head(10)



---First 10 rows---


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,3,11,0
5,7446,1967,Master,Together,62513.0,0,1,2013-09-09,16,520,...,6,0,0,0,0,0,0,3,11,0
6,965,1971,Graduation,Divorced,55635.0,0,1,2012-11-13,34,235,...,6,0,0,0,0,0,0,3,11,0
7,6177,1985,PhD,Married,33454.0,1,0,2013-05-08,32,76,...,8,0,0,0,0,0,0,3,11,0
8,4855,1974,PhD,Together,30351.0,1,0,2013-06-06,19,14,...,9,0,0,0,0,0,0,3,11,1
9,5899,1950,PhD,Together,5648.0,1,1,2014-03-13,68,28,...,20,1,0,0,0,0,0,3,11,0


In [4]:
print("---Data infos---")
marketing_df.info()

---Data infos---
<class 'pandas.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID                   2240 non-null   int64         
 1   Year_Birth           2240 non-null   int64         
 2   Education            2240 non-null   str           
 3   Marital_Status       2240 non-null   str           
 4   Income               2216 non-null   float64       
 5   Kidhome              2240 non-null   int64         
 6   Teenhome             2240 non-null   int64         
 7   Dt_Customer          2240 non-null   datetime64[us]
 8   Recency              2240 non-null   int64         
 9   MntWines             2240 non-null   int64         
 10  MntFruits            2240 non-null   int64         
 11  MntMeatProducts      2240 non-null   int64         
 12  MntFishProducts      2240 non-null   int64         
 13  MntSweetProducts     2240 n

### Data dictionary

| Column | Description | Type | Role |
|---|---|---|---|
| `ID` | Customer's unique identifier | id | excluded |
| `Year_Birth` | Customer's year of birth | numerical | feature |
| `Education` | Customer's level of education | categorical | feature |
| `Marital_Status` | Customer's marital status | categorical | feature |
| `Income` | Customer's yearly household income | numerical | feature |
| `Kidhome` | Number of small children in the household | numerical | feature |
| `Teenhome` | Number of teenagers in the household | numerical | feature |
| `Dt_Customer` | Date of the customer's enrolment with the company | date | feature (after transformation) |
| `Recency` | Number of days since the last purchase | numerical | feature |
| `MntWines` | Amount spent on wine in the last 2 years | numerical | feature |
| `MntFruits` | Amount spent on fruits in the last 2 years | numerical | feature |
| `MntMeatProducts` | Amount spent on meat in the last 2 years | numerical | feature |
| `MntFishProducts` | Amount spent on fish in the last 2 years | numerical | feature |
| `MntSweetProducts` | Amount spent on sweets in the last 2 years | numerical | feature |
| `MntGoldProds` | Amount spent on gold products in the last 2 years | numerical | feature |
| `NumDealsPurchases` | Number of purchases made with a discount | numerical | feature |
| `NumWebPurchases` | Number of purchases made through the website | numerical | feature |
| `NumCatalogPurchases` | Number of purchases made using a catalogue | numerical | feature |
| `NumStorePurchases` | Number of purchases made directly in stores | numerical | feature |
| `NumWebVisitsMonth` | Number of visits to the website in the last month | numerical | feature |
| `AcceptedCmp1` … `AcceptedCmp5` | 1 if the customer accepted the offer in campaign 1…5, 0 otherwise | binary | feature (past campaigns) |
| `Complain` | 1 if the customer complained in the last 2 years | binary | feature |
| `Z_CostContact` | Cost of contacting a customer | constant | dropped (business parameter) |
| `Z_Revenue` | Revenue when a customer accepts the offer | constant | dropped (business parameter) |
| `Response` | 1 if the customer accepted the offer in the **last** campaign, 0 otherwise | binary | **target** |

> **Note:** `Dt_Customer` will not be used raw; it will be converted into a customer tenure feature in notebook 03.


### Target distribution

In [5]:
print("Target distribution ")
print(marketing_df['Response'].value_counts(normalize=True) * 100)

Target distribution 
Response
0    85.089286
1    14.910714
Name: proportion, dtype: float64


**Observation:**
- The classes are imbalanced, in the last campaign, just around 15% of the clients accepted the offer the other 85% refused
- We should take that into consideration during training 


**Decision:**
- The accuracy will not be used as an evaluation metric
- Every split (train / validation / test, CV folds) will be **stratified** on `Response`.
- A random model has a PR-AUC ≈ 0.149: this is the floor any model must beat in order to be the best model.

### Data quality

In [6]:
print("---Statistical infos about data---")
marketing_df.describe()

---Statistical infos about data---


,ID,Year_Birth,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
count,2240.000000,2240.000000,2216.000000,2240.000000,2240.000000,2240,2240.000000,2240.000000,2240.000000,2240.000000,...,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.0,2240.0,2240.000000
mean,5592.159821,1968.805804,52247.251354,0.444196,0.506250,2013-07-10 10:01:42.857142,49.109375,303.935714,26.302232,166.950000,...,5.316518,0.072768,0.074554,0.072768,0.064286,0.013393,0.009375,3.0,11.0,0.149107
min,0.000000,1893.000000,1730.000000,0.000000,0.000000,2012-07-30 00:00:00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
25%,2828.250000,1959.000000,35303.000000,0.000000,0.000000,2013-01-16 00:00:00,24.000000,23.750000,1.000000,16.000000,...,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
50%,5458.500000,1970.000000,51381.500000,0.000000,0.000000,2013-07-08 12:00:00,49.000000,173.500000,8.000000,67.000000,...,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
75%,8427.750000,1977.000000,68522.000000,1.000000,1.000000,2013-12-30 06:00:00,74.000000,504.250000,33.000000,232.000000,...,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
max,11191.000000,1996.000000,666666.000000,2.000000,2.000000,2014-06-29 00:00:00,99.000000,1493.000000,199.000000,1725.000000,...,20.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.0,11.0,1.000000
std,3246.662198,11.984069,25173.076661,0.538398,0.544538,NaN,28.962453,336.597393,39.773434,225.715373,...,2.426645,0.259813,0.262728,0.259813,0.245316,0.114976,0.096391,0.0,0.0,0.356274


In [7]:
marketing_df.loc[marketing_df["Year_Birth"] < 1920, ["ID", "Year_Birth"]]

,ID,Year_Birth
192,7829,1900
239,11004,1893
339,1150,1899


In [8]:
marketing_df['ID'].duplicated().sum()

np.int64(0)

In [9]:
# Same customer under two different IDs? (duplicates on every column except ID) : if it's the case, it can produces data leakage 
n_dup_without_id = marketing_df.drop(columns=ID_COL).duplicated().sum()
print(f"Duplicated rows when ignoring ID: {n_dup_without_id}")

Duplicated rows when ignoring ID: 182


In [10]:
# Is 666666 an isolated value or are there several extreme incomes?
marketing_df.nlargest(5, "Income")[["ID", "Income"]]

,ID,Income
2233,9432,666666.0
617,1503,162397.0
687,1501,160803.0
1300,5336,157733.0
164,8475,157243.0


In [11]:
# Are customers with a missing Income different? (response rate, missing vs. not missing)
marketing_df.groupby(marketing_df["Income"].isna())[TARGET].agg(["count", "mean"]).rename(
    index={False: "Income known", True: "Income missing"}, columns={"mean": "response_rate"})

,count,response_rate
Income,,
Income known,2216,0.150271
Income missing,24,0.041667


In [12]:
for col in CATEGORICAL_COLS:
    print(f"--- {col} ---")
    print(marketing_df[col].value_counts(), "\n")

--- Education ---
Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: int64 

--- Marital_Status ---
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64 



### Notes
- We have just 2216 incomes out of 2240, so 24 missing values
- the max yearly household income (666666) is suspicious specially the median is around 51381
- Z_CostContact and Z_Revenue are constants because the std is 0
- 3 customers have a birth year before 1920 (ages above 113 at data collection), almost certainly data entry errors, because the subscription goes till 2014
- No duplicated Id's


### Additional observations
- **182 duplicated rows when ignoring `ID` (≈ 8% of the data):** the same customers appear under two different IDs. If one copy ends up in the training set and the other in the test set, the model is evaluated on customers it has already seen (data leakage, over-optimistic scores).
- **The income of 666666 is isolated:** the next highest incomes are around 157k–162k, so it is almost certainly a data entry error rather than a real group of very wealthy customers.
- **Missing `Income`:** response rate of 4.2% vs 15.0% for customers with a known income, but this is based on only 24 customers (1 positive), so the gap is not statistically reliable.
- **Odd categories:** `Marital_Status` contains `Alone` (3), `Absurd` (2) and `YOLO` (2), which are not meaningful marital statuses; `Basic` (54) is a rare level of `Education`.

### Decisions
- **Duplicates (ignoring `ID`) will be removed before the train/test split.** This is not a learned statistic, so it does not cause leakage, and it prevents the same customer from appearing in both train and test.
- **Nothing else is removed or imputed in this notebook.** Missing values, outliers (income, birth years) and rare categories will be handled in notebooks 02/03; any learned statistic (e.g. median imputation) will be fitted on the training set only.
- `Z_CostContact` and `Z_Revenue` will be dropped from the features but kept as business parameters (cost = 3, revenue = 11) for the profit analysis in notebook 08.
- `ID` stays in the DataFrame and is excluded only when building `X`.

### Leakage audit

Before any modelling, decide which columns are allowed as features.

| Column(s) | Risk | Decision |
|---|---|---|
| `ID` | Identifier, no signal | Excluded from features |
| `Response` | Target | Never in X |
| `AcceptedCmp1–5` | Past campaigns: legitimate **only if** they happened before the target campaign | Kept, assumption documented |
| `Mnt*`, `Num*`, `Recency`, `Complain` | Window "last 2 years": may overlap the target campaign | Kept, assumption documented |
| `Z_CostContact`, `Z_Revenue` | Constant | Dropped from features |
| Any learned statistic (median, scaler, encoder) | Computed on full data = leakage | Fitted on train only, inside a `Pipeline` |

In [13]:
# Descriptive check (not a model): response rate by past-campaign acceptance.
# A strong gap is expected in marketing; a near-perfect one would be a leakage red flag.
accepted_before = (marketing_df[PAST_CAMPAIGN_COLS].sum(axis=1) > 0)
marketing_df.groupby(accepted_before)[TARGET].agg(["count", "mean"]).rename(
    index={False: "never accepted before", True: "accepted >= 1 past campaign"},
    columns={"mean": "response_rate"})

,count,response_rate
never accepted before,1777,0.082161
accepted >= 1 past campaign,463,0.406048


**Observation:** customers who accepted at least one past campaign responded at **40.6%**, versus **8.2%** for the others (≈ 5× more). The link is strong but far from perfect (60% of past acceptors still refused), which is expected in marketing and not a leakage red flag.

**Decision:** past campaigns are kept as features. They also give a natural **business-rule baseline** for notebook 04: "target only customers who accepted a previous campaign".

---
## A/B test dataset

Users were randomly assigned to **`ad`** (saw the real advert, treatment) or **`psa`** (saw a public service announcement, control). Outcome: `converted`.

Here we only describe the experiment. Hypothesis tests, confidence intervals and power come in notebook 08.

In [14]:
ab_df = load_ab_test()
print(ab_df.shape)
ab_df.head()

(588101, 6)


,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,1069124,ad,0,130,Monday,20
1,1119715,ad,0,93,Tuesday,22
2,1144181,ad,0,21,Tuesday,18
3,1435133,ad,0,355,Tuesday,10
4,1015700,ad,0,276,Friday,14


In [15]:
ab_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   user_id        588101 non-null  int64
 1   test_group     588101 non-null  str  
 2   converted      588101 non-null  int64
 3   total_ads      588101 non-null  int64
 4   most_ads_day   588101 non-null  str  
 5   most_ads_hour  588101 non-null  int64
dtypes: int64(4), str(2)
memory usage: 26.9 MB


In [16]:
# One user = one unit of randomisation: each user_id should appear once
print("Duplicated user_id:", ab_df["user_id"].duplicated().sum())

Duplicated user_id: 0


In [17]:
group_summary = ab_df.groupby(AB_GROUP_COL).agg(
    n_users=(AB_TARGET, "size"),
    conversions=(AB_TARGET, "sum"),
    conversion_rate=(AB_TARGET, "mean"),
)
group_summary["share_of_users"] = group_summary["n_users"] / group_summary["n_users"].sum()
group_summary

,n_users,conversions,conversion_rate,share_of_users
test_group,,,,
ad,564577,14423,0.025547,0.96
psa,23524,420,0.017854,0.04


**Observation:**
- 588,101 users, no duplicated `user_id`: each user is a single unit of randomisation.
- **96% `ad` / 4% `psa`**: a very unbalanced allocation (564,577 vs 23,524 users).
- Raw conversion rates: **2.55%** (`ad`) vs **1.79%** (`psa`). The ad group converts more, but no conclusion is drawn before the statistical tests of notebook 08.

**Points to keep in mind for notebook 08:**
- The split between groups is very unbalanced (expected ≈ 96% `ad` / 4% `psa`). This is common (companies keep the control group small), but we need the **intended** allocation to run a *Sample Ratio Mismatch* (SRM) check. If the intended split was 50/50, the randomisation is broken.
- The small control group will drive the width of the confidence interval and the statistical power.
- `total_ads` is measured **during** the experiment (post-treatment variable): it can describe the experiment, never be used to "adjust" the effect.

---
## Conclusion

### Decisions taken
| # | Decision | Why |
|---|---|---|
| 1 | Binary classification on `Response`, probability output | Ranking + adjustable threshold |
| 2 | PR-AUC as primary metric, lift and profit as business metrics | Imbalanced target (14.9%), business wants ROI |
| 3 | Stratified train / validation / test split, stratified CV | Keep the 14.9% positive rate in every subset |
| 4 | No imputation / outlier treatment / encoding before the split | Avoid data leakage |
| 5 | Drop `ID`, `Z_CostContact`, `Z_Revenue` from the features | No signal / constant |
| 6 | A/B dataset analysed separately, SRM check before any test | Validity of the experiment |
| 7 | Remove the 182 duplicated customers (ignoring `ID`) before the split | Avoid the same customer in train and test |

### Open questions for notebook 02 (EDA)
1. How to handle the 24 missing `Income` values (median imputation fitted on train?) and the extreme incomes?
2. Remove or correct the 3 impossible birth years?
3. How to regroup the rare / odd categories of `Marital_Status` and `Education`?
4. Which features have the strongest link with `Response`?
5. Are the spending features strongly correlated (multicollinearity for Logistic Regression)?
6. A/B test: what was the intended allocation ratio?

**Next step:** split the data at the start of notebook 02 and run the EDA on the **training set only**.